# [P3] Q2 C + D: analytical questions and outliers

**Owner:** Part 3  
**Temporary part notebook.** At integration it is merged into `Q2_laptop_analysis.ipynb` and this file is deleted.


## Before writing any code

Tasks: `t_q2c` (questions, RAM vs price required), `t_q2d` (outliers).

Decisions to read on the team board first (board ids): `qs tr ram_rep ram_m viz g_test g_n` (C), `o_def o_other o_treat` (D), and the shared `st_extra`.

Do not write code for a task until its decisions are made. Mark your cells with the part tag so the merge is easy.


# [P3] Q2 C: Analytical questions

We use the cleaned table from Part 2 (`load_clean()`) and investigate three questions about the laptop dataset. The decisions below were made on the team board.

| Decision | What we chose |
|---|---|
| `qs` | Three questions: RAM and price (required), SSD against HDD, and brand premium. Together they use two kinds of analysis: a relationship between numeric variables, and comparison of groups. |
| `tr` | We analyse the natural log of the price (`log_price`) and report results back in euro and in percent. |
| `ram_rep` | RAM is numeric as log2(RAM) for correlation and regression, and categorical in box plots. Levels with fewer than 20 laptops are flagged or merged. |
| `ram_m` | Spearman and Pearson (log-log), plus an OLS regression with controls (CPU tier, laptop type, SSD, screen resolution, brand). |
| `viz` | RAM against price as a jittered scatter with a median line per level and a log y axis. |
| `g_test` | Mann-Whitney or Kruskal-Wallis, with an effect size (rank-biserial), a bootstrap confidence interval and Holm correction. |
| `g_n` | A group needs at least 20 laptops to be compared. |
| `q_tpl` | Each question has five fixed headings: why it is interesting, which variables and rows, which method, figure, interpretation. |

## C0. Setup and check of the variables we need

Before any analysis we load the cleaned table and check that it has the columns we rely on. Code that runs is not proof that the table is right, so we use `assert` to stop if a column is missing, if a value is missing, or if the price is not positive. We also add `RAM_log2` (decision `ram_rep`): RAM doubles in steps (4, 8, 16, 32...), and log2 turns one step into one doubling.

In [ ]:
# [P3] C0: setup
# HE: ייבוא הספריות. load_clean היא הפונקציה של טל שקוראת את הקובץ ומנקה אותו בקריאה אחת.
import numpy as np
import pandas as pd
from q2_clean import load_clean

# HE: לוקחים את הטבלה הנקייה. היא מכילה את 15 העמודות המקוריות ועוד 17 חדשות.
df = load_clean()
print(df.shape)

In [ ]:
# [P3] C0: contract check and RAM_log2
# HE: אלה העמודות שאנחנו משתמשים בהן בשאלות. אם טל ישנה שם של עמודה, הבדיקה תיכשל ונדע מיד.
needed = ["Price (Euro)", "log_price", "RAM (GB)", "CPU_tier", "TypeName", "has_SSD",
          "Screen_pixels", "Company", "Company_grouped", "Storage_type", "near_duplicate"]
missing_cols = [c for c in needed if c not in df.columns]
assert not missing_cols, f"columns missing from load_clean(): {missing_cols}"
# HE: בודקים שאין ערכים חסרים בעמודות האלה, ושהמחיר חיובי (אחרת הלוג לא מוגדר).
assert df[needed].notna().all().all(), "missing values in the columns we need"
assert (df["Price (Euro)"] > 0).all()

# HE: log2 של RAM: מדרגה אחת (הכפלה) נעשית "1" בציר.
df["RAM_log2"] = np.log2(df["RAM (GB)"])

# HE: בדיקה על ערכים שהתוצאה שלהם ידועה מראש: 2 GB = 2^1, 8 GB = 2^3, 64 GB = 2^6.
known = {2: 1, 8: 3, 64: 6}
for gb, expected in known.items():
    assert (df.loc[df["RAM (GB)"] == gb, "RAM_log2"] == expected).all()
print("columns present, no missing values, RAM_log2 correct on the known cases")

### RAM levels and how many laptops each has

Decision `ram_rep` says that levels with fewer than 20 laptops must be flagged or merged, because a group that small gives an unstable median and a very wide interval. Here we only **count and flag**. Whether we merge the rare levels into one group (for example "24 GB or more") is decided when we draw the box plot.

In [ ]:
# [P3] C0: RAM levels with counts
# HE: כמה מחשבים בכל רמת RAM, ובנוסף עמודה שמסמנת רמה עם פחות מ-20 מחשבים (הסף של ההחלטה g_n).
ram_levels = (df.groupby("RAM (GB)")
                .agg(n=("RAM (GB)", "size"), median_price=("Price (Euro)", "median"))
                .assign(fewer_than_20=lambda t: t["n"] < 20))
print(ram_levels)
assert ram_levels["n"].sum() == len(df)

# Question 1: RAM and price

## 1. Why is it interesting?

Someone choosing a laptop wants to know whether more RAM costs more, and how much. But RAM does not come alone: laptops with a lot of RAM usually also have a stronger processor, a more expensive brand and a different laptop type. So a link between RAM and price in the raw data may be partly or mostly caused by these other characteristics. This is the difference between correlation and causation that the project instructions ask us to respect.

## 2. Which variables and observations?

- **Variables:** `RAM_log2` (RAM as log2, decision `ram_rep`) and `log_price` (decision `tr`). We also keep the raw RAM in GB and the raw price in euro, only to compare.
- **Observations:** all 1,275 laptops. The 48 near-duplicate rows (same configuration, different price) are kept in the main analysis (decision `q_dups`). They are not independent observations, so near the end we repeat the key results without them.

## 3. Method

Decision `ram_m`: first the raw association (this step), then an OLS regression with controls (next step).

### 3a. The raw association

We report three coefficients. **Pearson** measures a straight-line relation, so we compute it on the log-log scale (the scale chosen in `tr` and `ram_rep`) and, for comparison, on the raw values. **Spearman** uses only the ranks, so it does not depend on the scale and is not driven by extreme values.

In [ ]:
# [P3] C1: raw RAM-price association
# HE: scipy.stats מחשבת את מקדמי המתאם. pearsonr = מתאם ליניארי, spearmanr = מתאם דרגות.
from scipy import stats

ram_gb = df["RAM (GB)"]
price = df["Price (Euro)"]

# HE: Pearson על הערכים הגולמיים, ועל לוג־לוג (log2 של RAM מול לוג טבעי של המחיר).
pearson_raw = stats.pearsonr(ram_gb, price).statistic
pearson_loglog = stats.pearsonr(df["RAM_log2"], df["log_price"]).statistic
# HE: Spearman על הערכים הגולמיים. הוא עובד על דרגות בלבד.
spearman = stats.spearmanr(ram_gb, price).statistic

# HE: בדיקה ראשונה: Spearman לא צריך להשתנות כשמחליפים לסקאלת לוג, כי לוג שומר על הסדר.
spearman_log = stats.spearmanr(df["RAM_log2"], df["log_price"]).statistic
assert abs(spearman - spearman_log) < 1e-12
# HE: בדיקה שנייה: Pearson גם בדרך בלתי תלויה, עם numpy.
assert abs(pearson_loglog - np.corrcoef(df["RAM_log2"], df["log_price"])[0, 1]) < 1e-12

# HE: טבלת תוצאות קטנה עם שמות ברורים, כדי ששחר יוכל לצטט מהן בדוח.
ram_price_raw = pd.Series({
    "Pearson, raw (GB vs euro)": pearson_raw,
    "Pearson, log-log": pearson_loglog,
    "Spearman (same on raw and log scale)": spearman,
}).round(3)
print(ram_price_raw)

### 3b. Regression with controls

The raw coefficients above mix the effect of RAM with the effect of everything that travels with RAM. To separate them we fit an OLS regression of `log_price` on `RAM_log2` and add the controls one group at a time (decision `ram_m`): CPU tier, laptop type, SSD, screen resolution and brand.

How to read the result: because the price is in log and RAM is in log2, the coefficient of `RAM_log2` is the change in log price for **one doubling of RAM**. We convert it to a percent change with `exp(b) - 1`. If the coefficient falls a lot after adding controls, much of the raw association was due to the other characteristics.

Choices we make here, and state as assumptions:
- Resolution enters as `Screen_Mpix`, the number of pixels in millions (a straight-line effect is assumed).
- Brand uses `Company_grouped` (companies with fewer than 10 laptops are in `Other`), and the baseline brand is Acer, the first alphabetically.
- The confidence interval is a bootstrap interval (1,000 resamples of the rows, fixed seed), in line with decision `g_test`. It is not a causal statement.

`statsmodels` is not in our `requirements.txt`, so the regression is written with `numpy`. Since we write it ourselves, we test it first on inputs whose answer we know.

In [ ]:
# [P3] C2: small OLS helpers and their tests
# HE: מסדרים את העמודות לטבלת מדדים: עמודות מספריות כמו שהן, וקטגוריות הופכות לעמודות 0/1 (dummies).
# HE: drop_first=True מוריד קטגוריה אחת לכל משתנה, והיא נעשית נקודת הייחוס (אחרת יש תלות מלאה בין העמודות).
def design_matrix(data, numeric, categorical):
    X = data[numeric].astype(float)
    if categorical:
        X = pd.concat([X, pd.get_dummies(data[categorical], drop_first=True, dtype=float)], axis=1)
    X.insert(0, "const", 1.0)
    return X

# HE: רגרסיה לינארית בשיטת הריבועים הפחותים. lstsq פותרת את המשוואה ומחזירה את המקדמים.
def ols_fit(X, y):
    beta = np.linalg.lstsq(X.to_numpy(), y.to_numpy(), rcond=None)[0]
    resid = y.to_numpy() - X.to_numpy() @ beta
    r2 = 1 - (resid ** 2).sum() / ((y.to_numpy() - y.mean()) ** 2).sum()
    return pd.Series(beta, index=X.columns), r2

# HE: רווח סמך בוטסטראפ: דוגמים שורות עם החזרה, מחשבים מחדש את המקדם, ולוקחים את האחוזונים 2.5 ו-97.5.
def bootstrap_ci(X, y, column, n_boot=1000, seed=2026):
    rng = np.random.default_rng(seed)
    Xv, yv = X.to_numpy(), y.to_numpy()
    j = list(X.columns).index(column)
    draws = np.empty(n_boot)
    for b in range(n_boot):
        idx = rng.integers(0, len(yv), len(yv))
        draws[b] = np.linalg.lstsq(Xv[idx], yv[idx], rcond=None)[0][j]
    return np.percentile(draws, [2.5, 97.5])

# HE: בדיקה 1: נתונים שבהם y = 1 + 2x בדיוק. המקדמים חייבים לצאת 1 ו-2, ו-R2 חייב לצאת 1.
toy = pd.DataFrame({"x": [0.0, 1.0, 2.0, 3.0, 4.0]})
toy["y"] = 1 + 2 * toy["x"]
toy_beta, toy_r2 = ols_fit(design_matrix(toy, ["x"], []), toy["y"])
assert np.allclose(toy_beta.to_numpy(), [1, 2]) and abs(toy_r2 - 1) < 1e-12

# HE: בדיקה 2, על הנתונים האמיתיים: במודל עם משתנה אחד, השיפוע שווה למתאם כפול היחס בין סטיות התקן.
beta0, _ = ols_fit(design_matrix(df, ["RAM_log2"], []), df["log_price"])
slope_from_pearson = pearson_loglog * df["log_price"].std() / df["RAM_log2"].std()
assert abs(beta0["RAM_log2"] - slope_from_pearson) < 1e-9
print("OLS helpers pass both tests; simple slope =", round(beta0["RAM_log2"], 3))

Now the five models. Model 0 has no controls and is the raw association written as a regression. Each next model adds one group of controls to the previous one.

In [ ]:
# [P3] C2: RAM coefficient as controls are added
# HE: Screen_Mpix = פיקסלים במיליונים, כדי שהמקדם יהיה בסדר גודל קריא.
df["Screen_Mpix"] = df["Screen_pixels"] / 1e6

# HE: כל מודל מוסיף קבוצת בקרות אחת למודל הקודם. השם של הרשימה: (עמודות מספריות, עמודות קטגוריאליות).
models = {
    "0 RAM only":                   (["RAM_log2"], []),
    "1 + CPU tier":                 (["RAM_log2"], ["CPU_tier"]),
    "2 + laptop type":              (["RAM_log2"], ["CPU_tier", "TypeName"]),
    "3 + SSD":                      (["RAM_log2", "has_SSD"], ["CPU_tier", "TypeName"]),
    "4 + resolution":               (["RAM_log2", "has_SSD", "Screen_Mpix"], ["CPU_tier", "TypeName"]),
    "5 + brand":                    (["RAM_log2", "has_SSD", "Screen_Mpix"], ["CPU_tier", "TypeName", "Company_grouped"]),
}

rows = []
for name, (numeric, categorical) in models.items():
    X = design_matrix(df, numeric, categorical)
    beta, r2 = ols_fit(X, df["log_price"])
    lo, hi = bootstrap_ci(X, df["log_price"], "RAM_log2")
    # HE: מתרגמים מקדם בלוג לאחוזי שינוי: exp(b) - 1. זה השינוי במחיר להכפלה אחת של ה-RAM.
    pct = lambda b: (np.exp(b) - 1) * 100
    rows.append({"model": name, "coef_per_doubling": beta["RAM_log2"],
                 "pct_change": pct(beta["RAM_log2"]), "ci_low_pct": pct(lo), "ci_high_pct": pct(hi), "R2": r2})

ram_price_adjusted = pd.DataFrame(rows).set_index("model").round(3)
print(ram_price_adjusted)
# HE: בדיקת שפיות: מודל 0 הוא אותה רגרסיה כמו הבדיקה למעלה.
assert abs(ram_price_adjusted.loc["0 RAM only", "coef_per_doubling"] - beta0["RAM_log2"]) < 1e-3

## 4. Figure: RAM against price

Decision `viz`: a scatter plot with a little jitter, a median line over the RAM levels, and a log price axis. RAM takes only nine values, so without jitter the 613 laptops with 8 GB would sit on one vertical line and hide each other. The jitter is only for drawing (horizontal up to 0.12 and vertical up to 1 percent); all calculations use the true values.

**Rare RAM levels (decision `ram_rep`, with `g_n`).** Four RAM levels have fewer than 20 laptops: 2 GB (16), 24 GB (3), 32 GB (17) and 64 GB (1). A median of one or three laptops is not an estimate, and a reader could take it for a real trend. We therefore **merge 24, 32 and 64 GB into one group, "24+" (21 laptops)**, and **flag 2 GB (16 laptops)** as too small. The choice was made by group size, not by looking at the result. No laptop is removed: every laptop is still a dot in the figure, and the correlations and the regression use `RAM_log2` for all 1,275 rows. What we lose is detail at the top end, where the median price rises between 24, 32 and 64 GB.

Figure rules (decision `g_style`): axis names with units, a readable font (12 pt), and a short caption under the figure. The same price range will be used in the box plot, so that the two figures can be compared.

In [ ]:
# [P3] C3: RAM groups and the numbers behind the figure
# HE: מאחדים את 24, 32 ו-64 לקבוצה אחת "24+" לפי ההחלטה. שאר הרמות נשארות כמו שהן.
df["RAM_group"] = np.where(df["RAM (GB)"] >= 24, "24+", df["RAM (GB)"].astype(str))
order = ["2", "4", "6", "8", "12", "16", "24+"]
MIN_N = 20   # HE: הסף מההחלטה g_n

# HE: לכל קבוצה: כמה מחשבים, חציון המחיר, והטווח שהיא תופסת על ציר ה-log2.
groups = (df.groupby("RAM_group")
            .agg(n=("RAM_log2", "size"), median_price=("Price (Euro)", "median"), mean_price=("Price (Euro)", "mean"),
                 x_min=("RAM_log2", "min"), x_max=("RAM_log2", "max"))
            .loc[order])
groups["stable"] = groups["n"] >= MIN_N

# HE: בדיקות: אף מחשב לא אבד, ולקבוצה המאוחדת יש בדיוק 21 מחשבים (3 + 17 + 1).
assert groups["n"].sum() == len(df)
assert groups.loc["24+", "n"] == 21
print(groups.round(0))

In [ ]:
# [P3] C3: figure of RAM against price
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.ticker import NullFormatter

# HE: גופן קריא (החלטה g_style). הגרפים נשמרים בתיקיית figures בשורש המאגר (200 dpi, החלטה g_figs).
plt.rcParams.update({"font.size": 12, "axes.labelsize": 12})
FIG_DIR = Path("..") / "figures" if Path("../data").exists() else Path("figures")
FIG_DIR.mkdir(exist_ok=True)
PRICE_YLIM = (150, 7000)   # HE: אותם גבולות ציר גם בבוקספלוט, כדי שאפשר יהיה להשוות (החלטה g_style)

# HE: jitter רק לציור: הזזה אקראית קטנה (זרע קבוע, כדי שהגרף יהיה זהה בכל הרצה).
rng = np.random.default_rng(7)
x_plot = df["RAM_log2"] + rng.uniform(-0.12, 0.12, len(df))
y_plot = df["Price (Euro)"] * np.exp(rng.uniform(-0.01, 0.01, len(df)))

fig, ax = plt.subplots(figsize=(11, 6))
ax.scatter(x_plot, y_plot, s=14, alpha=0.35, color="#4C72B0", edgecolors="none")

# HE: קו חציון לכל קבוצה. קבוצה יציבה (n>=20): קו שחור רציף. קבוצה קטנה (2 GB): קו אדום מקווקו.
for name, g in groups.iterrows():
    color, style = ("black", "-") if g["stable"] else ("tab:red", "--")
    ax.hlines(g["median_price"], g["x_min"] - 0.2, g["x_max"] + 0.2, color=color, linestyle=style, linewidth=2.5)
    # HE: כתובת מחיר החציון בשקלי אירו מעל כל קו (רק הערך הספציפי, בלי קווי רשת נוספים).
    ax.annotate(f"€{g['median_price']:,.0f}", xy=((g["x_min"] + g["x_max"]) / 2, g["median_price"]),
                xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=10, fontweight="bold",
                color=color, bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=1.5))
# HE: קו שמחבר בין החציונים של הקבוצות היציבות בלבד, כדי לראות את המגמה.
stable = groups[groups["stable"]]
ax.plot((stable["x_min"] + stable["x_max"]) / 2, stable["median_price"], color="black", linewidth=1, alpha=0.6)

ax.set_yscale("log")
# HE: מבטלים את התוויות הקטנות בציר הלוגריתמי (כמו 6x10^3), כדי שלא יהיה עומס.
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_ylim(PRICE_YLIM)
ax.set_yticks([200, 300, 500, 1000, 2000, 3000, 5000])
ax.set_yticklabels(["200", "300", "500", "1,000", "2,000", "3,000", "5,000"])
# HE: ציר ה-x הוא log2 של RAM, אבל התוויות הן ה-GB עצמם, עם מספר המחשבים בכל רמה.
levels = sorted(df["RAM (GB)"].unique())
counts = df["RAM (GB)"].value_counts()
ax.set_xticks(np.log2(levels))
ax.set_xticklabels([f"{gb}\nn={counts[gb]}" for gb in levels])
ax.tick_params(axis="x", labelsize=10)
ax.set_xlabel("RAM (GB, log2 scale)")
ax.set_ylabel("Price (euro, log scale)")
ax.text(5.25, 2300, "24+ GB: pooled\nmedian (n=21)", fontsize=10)
ax.plot([], [], color="black", linewidth=2.5, label="median (group of 20 laptops or more)")
ax.plot([], [], color="tab:red", linestyle="--", linewidth=2.5, label="median of a group under 20 (2 GB: flagged)")
ax.legend(loc="upper left", fontsize=10)

fig.savefig(FIG_DIR / "q2_ram_vs_price_scatter.png", dpi=200, bbox_inches="tight")
plt.show()
assert (FIG_DIR / "q2_ram_vs_price_scatter.png").exists()

**Figure caption (saved as `figures/q2_ram_vs_price_scatter.png`).** Price against RAM for all 1,275 laptops, both axes on a log scale, with a little jitter for drawing. Black lines are medians of groups with 20 laptops or more, and the pooled "24+ GB" line covers 24, 32 and 64 GB (21 laptops). The number above each line is that group's median price in euro. The dashed red line is the median of the 2 GB group, which has only 16 laptops. The median price rises with RAM (from 498 euro at 4 GB to 1,895 euro at 16 GB and 3,073 euro for 24+ GB, see the table above). The spread inside each level is large, and neighbouring levels overlap, so RAM alone does not determine the price. This is why the next step compares the box plots and adjusts for other characteristics.

## 5. Figure: price by RAM level (box plot)

Decision `ram_rep`: besides the numeric RAM (log2) used in the correlation and the regression, we show RAM as a **category** in a box plot. A box plot does not assume that price rises in a straight line with RAM, so it shows each level on its own. It is a cross-check of the straight-line assumption.

How to read a box: the line inside is the median, the box covers the middle half of the prices (from the first to the third quartile), the whiskers reach the most extreme prices within 1.5 times that range, and dots beyond them are unusual values. We use the same groups as before: "24+" is 24, 32 and 64 GB together, and 2 GB is flagged because it has fewer than 20 laptops. We use exactly the same price range and the same price ticks as in the scatter figure (decision `g_style`: same axis limits when comparing), so the two figures can be compared. The boxes are evenly spaced because RAM is treated as a category here.

In [ ]:
# [P3] C4: quartiles and box plot of price by RAM group
# HE: הרבעונים של כל קבוצה, כדי שנוכל לצטט מספרים מהגרף בכיתוב.
quartiles = (df.groupby("RAM_group")["Price (Euro)"]
               .quantile([0.25, 0.5, 0.75]).unstack().loc[order])
quartiles.columns = ["q1", "median", "q3"]
print(quartiles.round(0))

nl = chr(10)   # HE: תו שורה חדשה, כדי לכתוב בתוויות את מספר המחשבים מתחת לקבוצה.
data_by_group = [df.loc[df["RAM_group"] == k, "Price (Euro)"] for k in order]

fig, ax = plt.subplots(figsize=(11, 6))
# HE: boxplot מצייר קופסה לכל קבוצה. patch_artist=True מאפשר לשנות את צבע הקופסאות.
bp = ax.boxplot(data_by_group, positions=range(len(order)), widths=0.5, patch_artist=True,
                flierprops=dict(marker="o", markersize=3, alpha=0.5))
for i, k in enumerate(order):
    # HE: קבוצה יציבה: כחול וקו רציף. קבוצה קטנה מ-20 (2 GB): אדום ומקווקו, כמו בגרף הפיזור.
    ok = groups.loc[k, "stable"]
    color = "#4C72B0" if ok else "tab:red"
    bp["boxes"][i].set(facecolor="white", edgecolor=color, linewidth=1.8, linestyle="-" if ok else "--")
    bp["medians"][i].set(color="black" if ok else "tab:red", linewidth=2.5)
    # HE: לכל קופסה שני שפמים ושני כובעים.
    for part in bp["whiskers"][2 * i: 2 * i + 2] + bp["caps"][2 * i: 2 * i + 2]:
        part.set(color=color, linewidth=1.4)
    bp["fliers"][i].set(markeredgecolor=color, markerfacecolor="none")
    # HE: מחיר החציון בשקלי אירו, ליד הקופסה (אותו כלל כמו בגרף הפיזור).
    ax.annotate(f"€{quartiles.loc[k, 'median']:,.0f}", xy=(i + 0.25, quartiles.loc[k, "median"]),
                xytext=(4, 0), textcoords="offset points", ha="left", va="center", fontsize=10,
                fontweight="bold", color="black" if ok else "tab:red",
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=1.5))

ax.set_yscale("log")
ax.yaxis.set_minor_formatter(NullFormatter())
ax.set_ylim(PRICE_YLIM)   # HE: אותם גבולות ואותם סימוני ציר כמו בגרף הפיזור
ax.set_yticks([200, 300, 500, 1000, 2000, 3000, 5000])
ax.set_yticklabels(["200", "300", "500", "1,000", "2,000", "3,000", "5,000"])
ax.set_xticks(range(len(order)))
ax.set_xticklabels([f"{k}{nl}n={groups.loc[k, 'n']}" for k in order])
ax.tick_params(axis="x", labelsize=10)
ax.set_xlabel("RAM group (GB)")
ax.set_ylabel("Price (euro, log scale)")
ax.plot([], [], color="#4C72B0", linewidth=1.8, label="group of 20 laptops or more")
ax.plot([], [], color="tab:red", linestyle="--", linewidth=1.8, label="group under 20 (2 GB: flagged)")
ax.legend(loc="upper left", fontsize=10)

fig.savefig(FIG_DIR / "q2_ram_vs_price_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()
assert (FIG_DIR / "q2_ram_vs_price_boxplot.png").exists()

In [ ]:
# [P3] C4: how much does the median price grow when RAM doubles?
# HE: היחס בין חציוני המחיר כשה-RAM מוכפל: מ-4 ל-8 GB, ומ-8 ל-16 GB. המספרים בכיתוב מגיעים מכאן.
median_by_group = quartiles["median"]
ratio_4_to_8 = median_by_group["8"] / median_by_group["4"]
ratio_8_to_16 = median_by_group["16"] / median_by_group["8"]
print(f"median price ratio, 4 -> 8 GB: {ratio_4_to_8:.2f}; 8 -> 16 GB: {ratio_8_to_16:.2f}")

**Figure caption (saved as `figures/q2_ram_vs_price_boxplot.png`).** Price for each RAM group, same price range and ticks as the scatter figure. The line in each box is the median (written next to it in euro), the box is the middle half of the prices, and dots are unusual values. The medians rise at every step, so the relation is monotone, but the boxes overlap a lot: the 6 GB box (549 to 677 euro) lies inside the 4 GB box (380 to 699 euro), and the 12 GB box (899 to 1,673 euro) covers most of the 8 GB box (847 to 1,450 euro). The step is not constant either: the median price is about 2.2 times higher at 8 GB than at 4 GB, and about 1.7 times higher at 16 GB than at 8 GB (see the ratios above). The dashed red box (2 GB) has 16 laptops and should be read with care. This agrees with the scatter figure and with using log2 of RAM in the regression.

## 6. Interpretation

**What we observed.** Laptops with more RAM cost more. The correlation between RAM and price is 0.74 (Pearson, raw values), 0.77 (Pearson, log-log) and 0.76 (Spearman). The median price rises at every RAM level (498 euro at 4 GB, 1,895 euro at 16 GB, 3,073 euro for 24+ GB), and without controls each doubling of RAM goes with a price about 90 percent higher (interval about 85 to 96 percent).

**What the analysis supports.** A large part of that association is shared with other characteristics. After controlling for CPU tier, laptop type, SSD, resolution and brand, the RAM coefficient falls from 0.641 to 0.301, which is a price about 35 percent higher per doubling (interval about 30 to 40 percent). The biggest drop comes from adding the CPU tier. So RAM is clearly associated with price even among similar laptops, but much less than the raw numbers suggest. Removing the 48 near-duplicate rows changes this estimate only slightly (about 36 percent).

**What we cannot conclude.** We cannot say that adding RAM to a laptop raises its price by 35 percent: this is an association between laptops, not an experiment. Characteristics we did not measure (processor generation, the exact graphics card, build quality) may travel with RAM and are not controlled. The model assumes the same percentage effect for all laptops, while the box plots show that the step is not constant (the median is 2.2 times higher at 8 GB than at 4 GB and 1.7 times higher at 16 GB than at 8 GB). The 2 GB and 24+ GB groups rest on few laptops. Finally, we do not know when or where the prices were collected.

# Question 2: Are laptops with an SSD more expensive than laptops with an HDD?

## 1. Why is it interesting?

For a consumer, an SSD is seen as a main upgrade, so it is natural to ask how much more a laptop with an SSD costs. But a price gap between SSD and HDD laptops is not necessarily the price of the SSD itself: SSDs appear mostly in newer and stronger laptops, which cost more for other reasons. So the raw gap is a description of the market, not the price of the drive.

## 2. Which variables and observations?

- **Variables:** `Storage_type` (built by Part 2 from the `Memory` column) and the price in euro. The comparison is rank-based, so it gives the same result on the raw price and on the log price.
- **Observations (decision made for this question):** we compare **SSD-only laptops with HDD-only laptops**. This is the cleanest contrast and one comparison. The laptops with both drives (`SSD + HDD`, 200 laptops) and those with Flash Storage (70) are shown in the figure for context but are **not tested**. The `Other storage` group (11 laptops) is below our minimum of 20 laptops per group (decision `g_n`) and is left out of the comparison.
- The 48 near-duplicate rows are kept in the main analysis (decision `q_dups`); at the end we repeat the key result without them.

## 3. Method

Decision `g_test`: a Mann-Whitney test, an effect size (the rank-biserial correlation) and a bootstrap confidence interval. With 1,275 laptops almost any difference is "significant", so the effect size and the interval matter more than the p-value. The Holm correction is for several comparisons at once; here there is a single comparison, so it changes nothing, and we will use it in the brand question.

The **rank-biserial correlation** is the probability that a random SSD laptop is more expensive than a random HDD laptop, minus the probability of the opposite. It goes from -1 to 1; 0 means no difference. We also report the ratio of the median prices, because the project reports results in euro and percent (decision `tr`).

In [ ]:
# [P3] C5: the groups, the effect size function and its tests
# HE: מפרידים את שתי הקבוצות (מחשבי SSD בלבד ומחשבי HDD בלבד) ובודקים שכל אחת עומדת בסף g_n.
ssd = df.loc[df["Storage_type"] == "SSD", "Price (Euro)"].to_numpy()
hdd = df.loc[df["Storage_type"] == "HDD", "Price (Euro)"].to_numpy()
assert min(len(ssd), len(hdd)) >= MIN_N
print("laptops: SSD-only", len(ssd), ", HDD-only", len(hdd))

# HE: rank-biserial: מדרגים את כל התצפיות יחד, מחשבים את U של הקבוצה הראשונה, והופכים אותו למקדם בין -1 ל-1.
def rank_biserial(a, b):
    ranks = stats.rankdata(np.concatenate([a, b]))
    u_a = ranks[:len(a)].sum() - len(a) * (len(a) + 1) / 2
    return 2 * u_a / (len(a) * len(b)) - 1

# HE: בדיקה 1: על דוגמה קטנה שהתשובה שלה ידועה. a=[3,4,5] מול b=[1,2,3]: ב-8 זוגות a גדול, בזוג אחד שווה, ב-0 קטן. 8/9.
toy = rank_biserial(np.array([3, 4, 5]), np.array([1, 2, 3]))
assert abs(toy - 8 / 9) < 1e-12

# HE: בדיקה 2, על הנתונים האמיתיים: חישוב ישיר על כל זוגות המחשבים (קבוצה א' יקרה מקבוצה ב' פחות ההפך), בלי דירוגים.
direct = (np.sum(ssd[:, None] > hdd[None, :]) - np.sum(ssd[:, None] < hdd[None, :])) / (len(ssd) * len(hdd))
r_obs = rank_biserial(ssd, hdd)
assert abs(r_obs - direct) < 1e-12

# HE: בדיקה 3: scipy נותנת את סטטיסטי U. המקדם שלנו חייב להתאים לו.
mw = stats.mannwhitneyu(ssd, hdd, alternative="two-sided")
assert abs(r_obs - (2 * mw.statistic / (len(ssd) * len(hdd)) - 1)) < 1e-12
print("effect size function passes three checks; rank-biserial =", round(r_obs, 3))

Now the confidence intervals. We use a **bootstrap**: we draw, with replacement, a new sample of the same size from each group, recompute the effect size and the ratio of medians, and repeat 2,000 times. The middle 95% of the results is the interval. The seed is fixed, so the numbers are the same in every run.

In [ ]:
# [P3] C5: bootstrap intervals and the result table
# HE: בוטסטראפ: דוגמים מחדש בנפרד מכל קבוצה (עם החזרה), ומחשבים את שני המדדים 2000 פעמים.
rng = np.random.default_rng(2026)
n_boot = 2000
r_boot = np.empty(n_boot)
ratio_boot = np.empty(n_boot)
for b in range(n_boot):
    a = rng.choice(ssd, len(ssd))
    c = rng.choice(hdd, len(hdd))
    r_boot[b] = rank_biserial(a, c)
    ratio_boot[b] = np.median(a) / np.median(c)

ci_r = np.percentile(r_boot, [2.5, 97.5])
ci_ratio = np.percentile(ratio_boot, [2.5, 97.5])
median_ssd, median_hdd = np.median(ssd), np.median(hdd)

ssd_vs_hdd = pd.Series({
    "n SSD-only": len(ssd), "n HDD-only": len(hdd),
    "median price SSD-only (euro)": median_ssd, "median price HDD-only (euro)": median_hdd,
    "median ratio SSD/HDD": median_ssd / median_hdd,
    "median ratio, 95% CI low": ci_ratio[0], "median ratio, 95% CI high": ci_ratio[1],
    "rank-biserial (SSD vs HDD)": r_obs,
    "rank-biserial, 95% CI low": ci_r[0], "rank-biserial, 95% CI high": ci_r[1],
    "P(random SSD laptop costs more than random HDD laptop)": (r_obs + 1) / 2,
    "Mann-Whitney p-value": mw.pvalue,
})
print(ssd_vs_hdd.round(3).to_string())
print("Mann-Whitney p-value, exact size:", f"{mw.pvalue:.1e}")

For context, here are all the storage types, described but not tested. The groups under 20 laptops are marked.

In [ ]:
# [P3] C5: descriptive table of every storage type
# HE: לכל סוג אחסון: כמה מחשבים, חציון והרבעונים, וסימון לקבוצות קטנות מ-20. הקבוצות האלה מוצגות בלבד ולא נבדקות.
storage_table = (df.groupby("Storage_type_grouped")["Price (Euro)"]
                   .agg(n="size", q1=lambda s: s.quantile(0.25), median="median", q3=lambda s: s.quantile(0.75))
                   .sort_values("n", ascending=False))
storage_table["under_20"] = storage_table["n"] < MIN_N
assert storage_table["n"].sum() == len(df)
print(storage_table.round(0))

## 4. Figure: price by storage type

A box plot of the price by storage type, split in two panels that share the price axis (decision `g_style`: same axis limits when comparing; the range and ticks are also the same as in the RAM figures). The **left panel is the main finding**: SSD-only against HDD-only, drawn larger and in blue. The **right panel is supplementary**: the groups shown for context only. The two groups we tested, SSD-only and HDD-only, are drawn in blue. The other groups are drawn in grey because they are shown for context only and were not tested. The group under 20 laptops (`Other storage`) is dashed and red, as before. The median price in euro is written next to each box.

In [ ]:
# [P3] C6: box plot of price by storage type, main finding and supplementary groups in two panels
main_groups = ["SSD", "HDD"]                                  # HE: שתי הקבוצות שנבדקו במבחן: הממצא העיקרי
context_groups = ["SSD + HDD", "Flash Storage", "Other storage"]   # HE: קבוצות להקשר בלבד, בלי מבחן

# HE: פונקציה שמציירת קופסאות לרשימת קבוצות בצירים נתונים. main=True מצייר אותן גדולות ובולטות יותר.
def draw_storage_boxes(ax, names, main):
    data = [df.loc[df["Storage_type_grouped"] == k, "Price (Euro)"] for k in names]
    bp = ax.boxplot(data, positions=range(len(names)), widths=0.55 if main else 0.45, patch_artist=True,
                    flierprops=dict(marker="o", markersize=4 if main else 3, alpha=0.5))
    for i, k in enumerate(names):
        n_k = int(storage_table.loc[k, "n"])
        med_k = storage_table.loc[k, "median"]
        # HE: אדום מקווקו = קבוצה קטנה מ-20. כחול = נבדק. אפור = הקשר בלבד.
        if n_k < MIN_N:
            color, style = "tab:red", "--"
        elif main:
            color, style = "#4C72B0", "-"
        else:
            color, style = "#7f7f7f", "-"
        line_w = 2.4 if main else 1.5
        text_color = color if color == "tab:red" else "black"
        bp["boxes"][i].set(facecolor="white", edgecolor=color, linewidth=line_w, linestyle=style)
        bp["medians"][i].set(color=text_color, linewidth=3 if main else 2.2)
        for part in bp["whiskers"][2 * i: 2 * i + 2] + bp["caps"][2 * i: 2 * i + 2]:
            part.set(color=color, linewidth=line_w - 0.4)
        bp["fliers"][i].set(markeredgecolor=color, markerfacecolor="none")
        ax.annotate(f"€{med_k:,.0f}", xy=(i + (0.28 if main else 0.25), med_k), xytext=(4, 0),
                    textcoords="offset points", ha="left", va="center", fontsize=13 if main else 10,
                    fontweight="bold", color=text_color,
                    bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=1.5))
    ax.set_xticks(range(len(names)))
    ax.set_xticklabels([f"{k}{nl}n={int(storage_table.loc[k, 'n'])}" for k in names],
                       fontsize=12 if main else 10)
    ax.set_xlim(-0.6, len(names) - 0.3)

# HE: שני צירים באותה שורה, עם ציר מחיר משותף (sharey): אותם גבולות בדיוק, לפי g_style.
fig, (ax_main, ax_ctx) = plt.subplots(1, 2, sharey=True, figsize=(12, 6), gridspec_kw={"width_ratios": [1, 1]})
draw_storage_boxes(ax_main, main_groups, main=True)
draw_storage_boxes(ax_ctx, context_groups, main=False)

for ax in (ax_main, ax_ctx):
    ax.set_yscale("log")
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_ylim(PRICE_YLIM)
    ax.set_yticks([200, 300, 500, 1000, 2000, 3000, 5000])
    ax.set_yticklabels(["200", "300", "500", "1,000", "2,000", "3,000", "5,000"])
ax_main.set_ylabel("Price (euro, log scale)")
ax_main.set_xlabel("Storage type")
ax_ctx.set_xlabel("Storage type")
ax_ctx.tick_params(axis="y", labelleft=False)

# HE: הכותרת מצטטת מספרים שחושבו למעלה (ssd_vs_hdd), לא מספרים שהוקלדו ביד.
ax_main.set_title("Main comparison (tested)" + nl +
                  f"median ratio {ssd_vs_hdd['median ratio SSD/HDD']:.2f}, "
                  f"rank-biserial {ssd_vs_hdd['rank-biserial (SSD vs HDD)']:.2f}",
                  fontsize=13, fontweight="bold", color="#4C72B0")
ax_ctx.set_title("Supplementary: context only, not tested", fontsize=11, color="#555555")
ax_ctx.plot([], [], color="#7f7f7f", linewidth=1.5, label="group of 20 laptops or more")
ax_ctx.plot([], [], color="tab:red", linestyle="--", linewidth=1.5, label="group under 20 laptops (flagged)")
ax_ctx.legend(loc="upper right", fontsize=9)
fig.tight_layout()

fig.savefig(FIG_DIR / "q2_ssd_vs_hdd_boxplot.png", dpi=200, bbox_inches="tight")
plt.show()
assert (FIG_DIR / "q2_ssd_vs_hdd_boxplot.png").exists()

**Figure caption (saved as `figures/q2_ssd_vs_hdd_boxplot.png`).** Price by storage type in two panels that share the price axis (euro, log scale), with the median written next to each box. Left: the main comparison. Right: supplementary groups. The two tested groups are in blue: SSD-only laptops (median 1,197 euro, middle half from 832 to 1,688 euro) and HDD-only laptops (median 598 euro, middle half from 449 to 798 euro). The two boxes do not overlap, because the top of the HDD box is below the bottom of the SSD box, and the SSD group has a much wider spread with more expensive laptops. Grey groups are context only, not tested: laptops with both drives have a higher median (1,398 euro) and Flash Storage laptops a much lower one (344 euro). We have not checked why. The dashed red group has only 11 laptops and is flagged. The gap between SSD and HDD is large, but it is a gap between kinds of laptops and not the price of the drive: this question is about the raw market, and the regression of question 1 shows that other characteristics matter.

## 5. Interpretation

**What we observed.** SSD-only laptops are much more expensive than HDD-only laptops. The median price is 1,197 euro against 598 euro, a ratio of about 2.0 (interval about 1.87 to 2.18). The rank-biserial correlation is 0.674 (interval 0.625 to 0.722): in about 84 percent of the pairs of one SSD-only and one HDD-only laptop, the SSD laptop is the more expensive one. The middle halves of the two groups do not overlap.

**What the analysis supports.** The gap is large and is not an artefact of a few extreme laptops, because the test works on ranks. It is almost unchanged when the 48 near-duplicate rows are removed (ratio 2.02, rank-biserial 0.667). The p-value is tiny, but with 1,275 laptops this says little; the size of the effect is what matters.

**What we cannot conclude.** We cannot say that an SSD makes a laptop twice as expensive. This is a gap between two kinds of laptops: SSDs are common in newer and stronger laptops, which cost more for other reasons, and in this question we did not control for them. We also did not test the laptops with both drives or with Flash Storage, and we did not check why the latter are so cheap. The storage type comes from parsing the `Memory` text, which is our reading of it (for example, Flash Storage is not counted as an SSD).

# Question 3: Is there a brand premium?

## 1. Why is it interesting?

Someone looking at the laptop market wants to know whether paying for some brands, such as Apple or MSI, means paying a premium over brands such as Acer or Lenovo. A real premium is a price gap between laptops with a **similar specification**. The raw gap can be misleading, because premium brands also sell stronger laptops. So we look at two things: the raw gap (this step), and the gap after controlling for the specification (next step).

## 2. Which variables and observations?

- **Variables:** the brand (`Company_grouped`, built by Part 2) and the price in euro.
- **Observations:** only brands with at least 20 laptops (decision `g_n`). The group `Other` is a union of 11 small companies, not a brand, so it is not used. The other brands are shown in a table for context.
- **Comparisons (decision made for this question):** four pairs, a premium brand against a reference brand: Apple against Acer, Apple against Lenovo, MSI against Acer and MSI against Lenovo. We chose the pairs from the question as it is phrased on the board; the other brands are not tested.
- The 48 near-duplicate rows are kept in the main analysis (decision `q_dups`).

## 3. Method

Decision `g_test`: for each pair a Mann-Whitney test, the rank-biserial effect size and a bootstrap confidence interval; the p-values are corrected with the **Holm correction**, because four tests are done on the same data. We also report the ratio of the median prices, so that results can be read in euro and percent. Positive rank-biserial means that the first brand (the premium brand) is more expensive.

Two cautions. The confidence intervals are **not** corrected for the four comparisons, only the p-values are. And Apple has only 21 laptops, so its intervals will be wide.

In [ ]:
# [P3] C7: brands with their sizes (context table)
# HE: לכל יצרן: מספר מחשבים, חציון והרבעונים. Other הוא איחוד של יצרנים קטנים ולכן לא יצרן אמיתי.
brand_table = (df.groupby("Company_grouped")["Price (Euro)"]
                 .agg(n="size", q1=lambda s: s.quantile(0.25), median="median", q3=lambda s: s.quantile(0.75))
                 .sort_values("median", ascending=False))
brand_table["usable"] = (brand_table["n"] >= MIN_N) & (brand_table.index != "Other")
assert brand_table["n"].sum() == len(df)
print(brand_table.round(0))

First the Holm correction, written by us and tested against a hand calculation. Holm sorts the p-values from small to large and multiplies the smallest by the number of tests, the next by that number minus one, and so on; then it makes sure the corrected values never go down, and caps them at 1.

In [ ]:
# [P3] C7: Holm correction and a function that compares two groups
def holm(p_values):
    p = np.asarray(p_values, dtype=float)
    order_idx = np.argsort(p)                      # HE: מסדרים את ה-p מהקטן לגדול
    m = len(p)
    adjusted = np.empty(m)
    running_max = 0.0
    for rank, idx in enumerate(order_idx):
        # HE: מכפילים ב-(m - מקום) ושומרים שהערכים המתוקנים לא יורדים לאורך הסדר.
        running_max = max(running_max, (m - rank) * p[idx])
        adjusted[idx] = min(1.0, running_max)
    return adjusted

# HE: בדיקה בחישוב ידני: p=[0.01, 0.04, 0.03] עם m=3 נותן [0.03, 0.06, 0.06].
assert np.allclose(holm([0.01, 0.04, 0.03]), [0.03, 0.06, 0.06])
# HE: p=[0.5, 0.9]: הקטן מוכפל ב-2 (=1.0), והשני נשאר לפחות 1.0, כי הערכים המתוקנים לא יורדים.  
assert np.allclose(holm([0.5, 0.9]), [1.0, 1.0])

# HE: משווה שתי קבוצות מחירים: חציונים, יחס חציונים, rank-biserial, רווחי סמך בבוטסטראפ ו-p של Mann-Whitney.
def compare_groups(a, b, seed, n_boot=2000):
    rng = np.random.default_rng(seed)
    r_boot = np.empty(n_boot)
    ratio_boot = np.empty(n_boot)
    for i in range(n_boot):
        a_i = rng.choice(a, len(a))
        b_i = rng.choice(b, len(b))
        r_boot[i] = rank_biserial(a_i, b_i)
        ratio_boot[i] = np.median(a_i) / np.median(b_i)
    r_low, r_high = np.percentile(r_boot, [2.5, 97.5])
    ratio_low, ratio_high = np.percentile(ratio_boot, [2.5, 97.5])
    return {"n_a": len(a), "n_b": len(b), "median_a": np.median(a), "median_b": np.median(b),
            "median_ratio": np.median(a) / np.median(b), "ratio_low": ratio_low, "ratio_high": ratio_high,
            "rank_biserial": rank_biserial(a, b), "r_low": r_low, "r_high": r_high,
            "p_raw": stats.mannwhitneyu(a, b, alternative="two-sided").pvalue}

# HE: בדיקה: על שתי קבוצות שאני יודע מראש את התשובה: a גדולה מ-b בכל הזוגות, ולכן rank-biserial=1 והיחס 10.
check = compare_groups(np.array([100.0, 110, 120, 130]), np.array([10.0, 11, 12, 13]), seed=1, n_boot=50)
assert check["rank_biserial"] == 1.0 and abs(check["median_ratio"] - 10) < 1e-9
print("Holm and compare_groups pass their checks")

In [ ]:
# [P3] C7: the four brand comparisons
# HE: ארבעת הזוגות: מותג יוקרתי מול מותג ייחוס. הקבוצות נשלפות מהעמודה Company.
pairs = [("Apple", "Acer"), ("Apple", "Lenovo"), ("MSI", "Acer"), ("MSI", "Lenovo")]
def prices_of(brand):
    return df.loc[df["Company"] == brand, "Price (Euro)"].to_numpy()

rows = []
for i, (premium, reference) in enumerate(pairs):
    a, b = prices_of(premium), prices_of(reference)
    assert min(len(a), len(b)) >= MIN_N       # HE: הסף מההחלטה g_n
    res = compare_groups(a, b, seed=[2026, i])
    res["pair"] = f"{premium} vs {reference}"
    rows.append(res)

brand_raw = pd.DataFrame(rows).set_index("pair")
# HE: תיקון Holm על ארבעת ערכי ה-p יחד.
brand_raw["p_holm"] = holm(brand_raw["p_raw"])
print(brand_raw[["n_a", "n_b", "median_a", "median_b", "median_ratio", "ratio_low", "ratio_high"]].round(2))
print()
print(brand_raw[["rank_biserial", "r_low", "r_high"]].round(3))
print()
print(brand_raw[["p_raw", "p_holm"]].map(lambda v: f"{v:.1e}"))

### 3b. The brand gap after controlling for the specification

The raw gap mixes the brand with everything that comes with it. To estimate the gap between laptops with a **similar specification**, we use the same regression as in question 1 (decision `ram_m`, model 5): `log_price` on RAM, SSD, resolution, CPU tier, laptop type and brand. The brand coefficients are differences in log price between brands, holding the other variables fixed. We convert a difference `b` to a price ratio with `exp(b)`. The baseline brand is Acer, so the Apple and MSI coefficients are already the gap to Acer; the gap to Lenovo is the difference between the two coefficients.

How to read it: an adjusted ratio of 1.2 means that, for laptops with the same RAM, SSD, resolution, CPU tier and laptop type, the premium brand is about 20 percent more expensive. This is a statement about a model, which assumes that each characteristic adds a fixed percentage to the price. Characteristics that we did not measure (the processor generation, build quality, the exact graphics card) are not controlled. The interval is a bootstrap interval over all rows, and, as before, it is not corrected for the four comparisons.

In [ ]:
# [P3] C8: brand coefficients from the regression with controls
brand_numeric = ["RAM_log2", "has_SSD", "Screen_Mpix"]
brand_categorical = ["CPU_tier", "TypeName", "Company_grouped"]
X_brand = design_matrix(df, brand_numeric, brand_categorical)
beta_brand, r2_brand = ols_fit(X_brand, df["log_price"])

# HE: כל צמד הוא צירוף ליניארי של המקדמים: Apple-Acer הוא המקדם של Apple (Acer הוא נקודת הייחוס), ו-Apple-Lenovo הוא ההפרש.
def contrast_weights(columns, plus, minus=None):
    w = pd.Series(0.0, index=columns)
    w["Company_grouped_" + plus] = 1.0
    if minus is not None:
        w["Company_grouped_" + minus] = -1.0
    return w

contrasts = {
    "Apple vs Acer": contrast_weights(X_brand.columns, "Apple"),
    "Apple vs Lenovo": contrast_weights(X_brand.columns, "Apple", "Lenovo"),
    "MSI vs Acer": contrast_weights(X_brand.columns, "MSI"),
    "MSI vs Lenovo": contrast_weights(X_brand.columns, "MSI", "Lenovo"),
}

# HE: בדיקה: מחליפים את נקודת הייחוס ל-Lenovo (מורידים את העמודה של Lenovo במקום של Acer) ומתאימים מחדש.
# HE: המקדם של Apple במודל הזה חייב להיות שווה להפרש Apple-Lenovo שחישבנו, אם הצירוף נכון.
brand_cols = [c for c in X_brand.columns if c.startswith("Company_grouped_")]
dummies = pd.get_dummies(df["Company_grouped"], dtype=float).drop(columns="Lenovo")
dummies.columns = ["Company_grouped_" + c for c in dummies.columns]
X_lenovo_base = pd.concat([X_brand.drop(columns=brand_cols), dummies], axis=1)
beta_lenovo_base, _ = ols_fit(X_lenovo_base, df["log_price"])
assert abs(beta_lenovo_base["Company_grouped_Apple"] - (contrasts["Apple vs Lenovo"] @ beta_brand)) < 1e-9
assert abs(beta_lenovo_base["Company_grouped_MSI"] - (contrasts["MSI vs Lenovo"] @ beta_brand)) < 1e-9
print("contrasts agree with a refit that uses Lenovo as the baseline")

In [ ]:
# [P3] C8: bootstrap intervals for the adjusted brand gaps
# HE: דוגמים שורות עם החזרה, מתאימים את הרגרסיה מחדש, ומחשבים את ארבעת ההפרשים. חוזרים 2000 פעמים.
rng = np.random.default_rng(2026)
n_boot = 2000
Xv, yv = X_brand.to_numpy(), df["log_price"].to_numpy()
W = np.column_stack([contrasts[name].to_numpy() for name in contrasts])
draws = np.empty((n_boot, len(contrasts)))
for b in range(n_boot):
    idx = rng.integers(0, len(yv), len(yv))
    draws[b] = np.linalg.lstsq(Xv[idx], yv[idx], rcond=None)[0] @ W

point = np.array([c @ beta_brand for c in contrasts.values()])
low, high = np.percentile(draws, [2.5, 97.5], axis=0)
brand_adjusted = pd.DataFrame({
    "raw_median_ratio": brand_raw["median_ratio"].to_numpy(),
    "adjusted_ratio": np.exp(point), "adj_ci_low": np.exp(low), "adj_ci_high": np.exp(high),
}, index=list(contrasts))
brand_adjusted["raw_ratio_ci_low"] = brand_raw["ratio_low"].to_numpy()
brand_adjusted["raw_ratio_ci_high"] = brand_raw["ratio_high"].to_numpy()
print(brand_adjusted.round(2))
print("R2 of the model:", round(r2_brand, 3))

## 4. Figure: the brand gap, raw and after controls

The figure has two panels. The **left panel is the main finding**: for each of the four pairs, the raw price ratio (ratio of medians, grey square) next to the ratio after controlling for the specification (blue circle), each with its 95% bootstrap interval. A ratio of 1 (dashed line) means no premium. The **right panel** shows the price of the four brands (box plots, same price range and ticks as the earlier figures); the brands we use as references are grey and the premium brands are blue.

In [ ]:
# [P3] C8: figure of the brand premium, raw and adjusted, with the prices of the four brands
fig, (ax_f, ax_b) = plt.subplots(1, 2, figsize=(14, 6), gridspec_kw={"width_ratios": [1.3, 1]})

# HE: הפאנל השמאלי (הממצא העיקרי): לכל זוג, יחס גולמי (ריבוע אפור) ויחס אחרי בקרה (עיגול כחול), עם רווחי הסמך.
labels = list(brand_adjusted.index)
y_pos = np.arange(len(labels))[::-1]       # HE: הזוג הראשון למעלה
for y, name in zip(y_pos, labels):
    r = brand_adjusted.loc[name]
    ax_f.errorbar(r["raw_median_ratio"], y + 0.13,
                  xerr=[[r["raw_median_ratio"] - r["raw_ratio_ci_low"]], [r["raw_ratio_ci_high"] - r["raw_median_ratio"]]],
                  fmt="s", color="#7f7f7f", capsize=4, markersize=8, linewidth=1.8)
    ax_f.errorbar(r["adjusted_ratio"], y - 0.13,
                  xerr=[[r["adjusted_ratio"] - r["adj_ci_low"]], [r["adj_ci_high"] - r["adjusted_ratio"]]],
                  fmt="o", color="#4C72B0", capsize=4, markersize=8, linewidth=2.2)
    ax_f.annotate(f"{r['raw_median_ratio']:.2f}", (r["raw_median_ratio"], y + 0.13), xytext=(0, 9),
                  textcoords="offset points", ha="center", fontsize=11, color="#555555")
    ax_f.annotate(f"{r['adjusted_ratio']:.2f}", (r["adjusted_ratio"], y - 0.13), xytext=(0, -20),
                  textcoords="offset points", ha="center", fontsize=12, fontweight="bold", color="#4C72B0")
ax_f.axvline(1, color="black", linestyle="--", linewidth=1)
ax_f.set_xscale("log")
ax_f.set_xticks([1, 1.5, 2, 3, 4])
ax_f.set_xticklabels(["1", "1.5", "2", "3", "4"])
ax_f.xaxis.set_minor_formatter(NullFormatter())
ax_f.set_xlim(0.8, 4.6)
ax_f.set_yticks(y_pos)
ax_f.set_yticklabels(labels)
ax_f.set_ylim(-0.8, 3.6)
ax_f.set_xlabel("Price ratio (premium brand / reference brand), log scale")
ax_f.set_title("Main finding: premium before and after controls", fontsize=13, fontweight="bold", color="#4C72B0")
ax_f.errorbar([], [], fmt="s", color="#7f7f7f", label="raw (ratio of medians)")
ax_f.errorbar([], [], fmt="o", color="#4C72B0", label="after controls (same specification)")
ax_f.legend(loc="upper center", bbox_to_anchor=(0.5, -0.17), fontsize=10, frameon=False)

# HE: הפאנל הימני: קופסאות המחיר של ארבעת היצרנים, באותו טווח ציר מחיר כמו בגרפים הקודמים.
box_brands = ["Acer", "Lenovo", "Apple", "MSI"]
bp = ax_b.boxplot([prices_of(k) for k in box_brands], positions=range(4), widths=0.5, patch_artist=True,
                  flierprops=dict(marker="o", markersize=3, alpha=0.5))
for i, k in enumerate(box_brands):
    color = "#4C72B0" if k in ("Apple", "MSI") else "#7f7f7f"
    bp["boxes"][i].set(facecolor="white", edgecolor=color, linewidth=1.8)
    bp["medians"][i].set(color="black", linewidth=2.5)
    for part in bp["whiskers"][2 * i: 2 * i + 2] + bp["caps"][2 * i: 2 * i + 2]:
        part.set(color=color, linewidth=1.4)
    bp["fliers"][i].set(markeredgecolor=color, markerfacecolor="none")
    med = np.median(prices_of(k))
    ax_b.annotate(f"€{med:,.0f}", xy=(i + 0.25, med), xytext=(4, 0), textcoords="offset points", ha="left",
                  va="center", fontsize=10, fontweight="bold",
                  bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=1.5))
ax_b.set_yscale("log")
ax_b.yaxis.set_minor_formatter(NullFormatter())
ax_b.set_ylim(PRICE_YLIM)
ax_b.set_yticks([200, 300, 500, 1000, 2000, 3000, 5000])
ax_b.set_yticklabels(["200", "300", "500", "1,000", "2,000", "3,000", "5,000"])
ax_b.set_xticks(range(4))
ax_b.set_xticklabels([f"{k}{nl}n={len(prices_of(k))}" for k in box_brands], fontsize=10)
ax_b.set_xlim(-0.6, 4.05)
ax_b.set_ylabel("Price (euro, log scale)")
ax_b.set_title("Price of the four brands", fontsize=11, color="#555555")
fig.tight_layout(w_pad=5)

fig.savefig(FIG_DIR / "q2_brand_premium.png", dpi=200, bbox_inches="tight")
plt.show()
assert (FIG_DIR / "q2_brand_premium.png").exists()

**Figure caption (saved as `figures/q2_brand_premium.png`).** Left: for each pair, the raw price ratio (ratio of medians, grey square) and the ratio after controlling for RAM, SSD, resolution, CPU tier and laptop type (blue circle), each with a 95% bootstrap interval; the dashed line at 1 means no premium. Right: price of the four brands, with the median in euro (Acer 572, Lenovo 899, Apple 1,340, MSI 1,599). The raw gaps are large (2.34 and 2.80 against Acer), but they shrink a lot after controls: Apple keeps a premium of about 33 percent over Acer and 20 percent over Lenovo, and the intervals stay above 1. For MSI the premium over Acer falls to 14 percent, and against Lenovo it is 1.03 with an interval (0.95 to 1.11) that includes 1, so we see no clear premium. Apple has only 21 laptops, so its intervals are wide.

## 5. Interpretation

**What we observed.** In the raw data, Apple (median 1,340 euro) and MSI (1,599 euro) are much more expensive than Acer (572 euro) and Lenovo (899 euro). The ratios of the medians are 2.34 and 2.80 against Acer, and 1.49 and 1.78 against Lenovo, and all four Holm-corrected p-values are below 0.001.

**What the analysis supports.** Much of the raw gap is not a brand effect. After controlling for RAM, SSD, resolution, CPU tier and laptop type, Apple still has a premium: about 33 percent over Acer (interval 1.20 to 1.51 as a ratio) and about 20 percent over Lenovo (1.08 to 1.34). MSI keeps a smaller premium over Acer (about 14 percent, 1.06 to 1.24), and against Lenovo there is no clear premium (ratio 1.03, interval 0.95 to 1.11, which includes 1). So the specification variables in our model explain most of MSI's raw gap against Lenovo, and a good part of Apple's. These conclusions do not change when the near-duplicate rows are removed.

**What we cannot conclude.** We cannot say that the brand name itself causes the premium. We only controlled for the characteristics listed above; the graphics card, processor generation, screen quality and build quality are not controlled and may differ between brands. The model assumes that each characteristic adds a fixed percentage to the price, and the result depends on the reference brand we compare against. Apple has only 21 laptops, so its intervals are wide. We chose these four pairs from the wording of the question, and the intervals are not corrected for making four comparisons (only the p-values are).

## Sensitivity: the key results without the near-duplicate rows

Part 2 found 48 rows that are identical to another row in every column except the price (24 pairs). They are not independent observations, and we kept them in the main analysis (decision `q_dups`). Here we remove all 48 and recompute the key numbers of the three questions, to see whether any conclusion depends on them. We show point estimates only.

In [ ]:
# [P3] Sensitivity: key results with and without the 48 near-duplicate rows
# HE: מוחקים את כל 48 השורות המסומנות (שני הצדדים של כל זוג) ומחשבים שוב את המספרים המרכזיים.
d2 = df.loc[~df["near_duplicate"]]
assert len(d2) == len(df) - 48

rows = []   # HE: כל שורה: שם המדד, הערך בכל הנתונים, והערך בלי הכפילויות
# HE: שאלה 1 (RAM): שני מקדמי המתאם ומקדם ה-RAM ברגרסיה המלאה (מודל 5).
rows.append(("Q1 RAM: Spearman", spearman, stats.spearmanr(d2["RAM (GB)"], d2["Price (Euro)"]).statistic))
rows.append(("Q1 RAM: Pearson log-log", pearson_loglog, stats.pearsonr(d2["RAM_log2"], d2["log_price"]).statistic))
num5, cat5 = models["5 + brand"]
beta5_d2, _ = ols_fit(design_matrix(d2, num5, cat5), d2["log_price"])
rows.append(("Q1 RAM: % price change per doubling (model 5)", ram_price_adjusted.loc["5 + brand", "pct_change"],
             (np.exp(beta5_d2["RAM_log2"]) - 1) * 100))
# HE: שאלה 2 (SSD מול HDD): יחס החציונים וגודל האפקט.
ssd2 = d2.loc[d2["Storage_type"] == "SSD", "Price (Euro)"].to_numpy()
hdd2 = d2.loc[d2["Storage_type"] == "HDD", "Price (Euro)"].to_numpy()
rows.append(("Q2 SSD/HDD median ratio", ssd_vs_hdd["median ratio SSD/HDD"], np.median(ssd2) / np.median(hdd2)))
rows.append(("Q2 SSD vs HDD rank-biserial", r_obs, rank_biserial(ssd2, hdd2)))
# HE: שאלה 3 (מותגים): הרגרסיה עם הבקרות מחושבת מחדש, ויחסי החציונים הגולמיים.
X_brand2 = design_matrix(d2, brand_numeric, brand_categorical)
assert list(X_brand2.columns) == list(X_brand.columns)      # HE: אותן עמודות בדיוק, אחרת הצירופים לא תקפים
beta_brand2, _ = ols_fit(X_brand2, d2["log_price"])
for name, w in contrasts.items():
    premium, reference = name.split(" vs ")
    raw_without = np.median(d2.loc[d2["Company"] == premium, "Price (Euro)"]) / np.median(d2.loc[d2["Company"] == reference, "Price (Euro)"])
    rows.append((f"Q3 {name}: raw median ratio", brand_raw.loc[name, "median_ratio"], raw_without))
    rows.append((f"Q3 {name}: adjusted ratio", brand_adjusted.loc[name, "adjusted_ratio"], np.exp(w @ beta_brand2)))

sensitivity = pd.DataFrame(rows, columns=["measure", "all rows", "without near-duplicates"]).set_index("measure")
sensitivity["difference"] = sensitivity["without near-duplicates"] - sensitivity["all rows"]
print(sensitivity.round(3).to_string())

# [P3] Q2 D: Unusual observations

The project instructions ask us to identify unusual or extreme observations, to ask whether an unusual value is necessarily an error, and **not** to remove outliers automatically: we must explain our decision and compare results before and after excluding them. The decisions below were made on the team board.

| Decision | What we chose |
|---|---|
| `o_def` | Two ways to find unusual prices: Tukey fences (1.5 times the IQR) on the log price, and residuals from a model of the specification (a laptop whose price is much higher or lower than its specification predicts). |
| `o_other` | Three checks that are not about price: physical fit (weight against screen size and type), rare configurations (RAM of 24 GB or more, a single CPU or GPU), and the spread of the price within the same configuration. |
| `o_treat` | Keep the unusual observations in the main analysis, and report a before-and-after comparison of the mean, the median, Pearson, Spearman and the regression coefficient. |

## 1. How do we define an outlier?

An outlier is an observation that is far from what is typical. "Far" depends on what we compare to, so we use two definitions. The first is **unconditional**: a price that is extreme relative to all prices. The second is **conditional**: a price that is extreme relative to what the laptop's specification would predict. A 6,000 euro gaming laptop may be extreme in the first sense but normal in the second; a laptop whose price does not fit its specification is the real suspect.

## 2. Tukey fences on the log price

A price is flagged when it is below `Q1 - 1.5 * IQR` or above `Q3 + 1.5 * IQR`, where Q1 and Q3 are the first and third quartiles and IQR is their difference. The rule uses quartiles, so the outliers themselves do not move the fences. We apply it to the log price (decision `tr`), because prices are multiplicative; the skewness of the raw price would otherwise make the expensive laptops look unusual just because of the scale.

In [ ]:
# [P3] D1: Tukey fences on log price
# HE: גדרות Tukey: Q1 פחות 1.5 כפול IQR, ו-Q3 ועוד 1.5 כפול IQR. ערך מחוץ לגדרות מסומן כחריג.
def tukey_fences(x, k=1.5):
    q1, q3 = np.percentile(x, [25, 75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

# HE: בדיקה על מקרה שהתשובה ידועה: ב-[1..9, 100] הרבעונים 3.25 ו-7.75, הגדרות -3.5 ו-14.5, ורק 100 חורג.
toy = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 100.0])
toy_low, toy_high = tukey_fences(toy)
assert abs(toy_low + 3.5) < 1e-9 and abs(toy_high - 14.5) < 1e-9
assert list(toy[(toy < toy_low) | (toy > toy_high)]) == [100.0]

fence_low, fence_high = tukey_fences(df["log_price"])
df["outlier_tukey_log"] = (df["log_price"] < fence_low) | (df["log_price"] > fence_high)

# HE: בדיקה שנייה: אותו חישוב עם pandas.quantile (דרך בלתי תלויה) חייב לתת אותו מספר חריגים.
q1_pd, q3_pd = df["log_price"].quantile([0.25, 0.75])
iqr_pd = q3_pd - q1_pd
assert df["outlier_tukey_log"].sum() == ((df["log_price"] < q1_pd - 1.5 * iqr_pd) | (df["log_price"] > q3_pd + 1.5 * iqr_pd)).sum()

print(f"fences on the log scale: {fence_low:.3f} to {fence_high:.3f}")
print(f"fences in euro: {np.exp(fence_low):,.0f} to {np.exp(fence_high):,.0f}")
print("laptops flagged:", int(df["outlier_tukey_log"].sum()), "of", len(df))
print(df.loc[df["outlier_tukey_log"], ["Company", "Product", "TypeName", "RAM (GB)", "Price (Euro)"]].to_string())

## 3. Laptops whose price does not fit their specification

For the conditional definition we fit a regression of `log_price` on the **specification only**: RAM (as log2), SSD, resolution (in megapixels), CPU tier and laptop type. The brand is not included (decision made for this section), so a brand premium will appear as a price higher than the specification predicts, and we will inspect it.

For each laptop the **residual** is the actual log price minus the log price that the model predicts. A positive residual means the laptop is more expensive than its specification suggests. We divide each residual by its standard deviation (the **standardized residual**; we use the version that also takes the leverage of the laptop into account), and flag a laptop when its absolute value is above 3 (decision `o_def`). If the residuals were normal, about 0.3 percent of the laptops, three or four of 1,275, would exceed 3 by chance.

A laptop can also be **influential**: removing it would change the regression a lot, even if its residual is not large. This is measured by Cook's distance, which combines the residual with the **leverage** (how unusual the laptop's specification is). We use the common rule of thumb `4 / n` to call a Cook's distance large; this threshold is our choice, not a board decision.

As before, we write the diagnostics ourselves and test them on inputs whose answer we know. In a regression of y on one variable x, the leverage of each point is `1/n + (x - mean(x))^2 / sum((x - mean(x))^2)`.

In [ ]:
# [P3] D2: leverage, standardized residuals and Cook's distance
# HE: leverage = האלכסון של מטריצת ה-"hat". דרך QR: אם X = QR אז h_i הוא סכום ריבועי השורה i של Q.
def leverage(X):
    q, _ = np.linalg.qr(X.to_numpy())
    return (q ** 2).sum(axis=1)

# HE: מחזיר שאריות, שאריות מתוקננות (internally studentized), leverage ו-Cook's distance לכל שורה.
def residual_diagnostics(X, y):
    beta, _ = ols_fit(X, y)
    Xv = X.to_numpy()
    n, k = Xv.shape
    assert np.linalg.matrix_rank(Xv) == k            # HE: אין תלות מלאה בין העמודות, אחרת leverage לא נכון
    resid = y.to_numpy() - Xv @ beta
    s2 = (resid ** 2).sum() / (n - k)                # HE: שונות השאריות
    h = leverage(X)
    standardized = resid / np.sqrt(s2 * (1 - h))
    cooks = resid ** 2 / (k * s2) * h / (1 - h) ** 2
    return resid, standardized, h, cooks

# HE: בדיקה 1: ברגרסיה על x=[1,2,3,4,5] ה-leverage ידוע מהנוסחה: 0.6, 0.3, 0.2, 0.3, 0.6.
toy = pd.DataFrame({"x": [1.0, 2.0, 3.0, 4.0, 5.0], "y": [1.0, 3.0, 2.0, 5.0, 4.0]})
h_toy = leverage(design_matrix(toy, ["x"], []))
assert np.allclose(h_toy, [0.6, 0.3, 0.2, 0.3, 0.6])
# HE: בדיקה 2: סכום ה-leverage שווה למספר הפרמטרים, בכל מודל (כאן 2: קבוע ושיפוע).
assert abs(h_toy.sum() - 2) < 1e-9
# HE: בדיקה 3: חישוב ישיר של h דרך pseudo-inverse (דרך בלתי תלויה) על המודל האמיתי.
X_spec = design_matrix(df, ["RAM_log2", "has_SSD", "Screen_Mpix"], ["CPU_tier", "TypeName"])
h_direct = np.diag(X_spec.to_numpy() @ np.linalg.pinv(X_spec.to_numpy()))
assert np.allclose(leverage(X_spec), h_direct)

resid, std_resid, lev, cooks = residual_diagnostics(X_spec, df["log_price"])
df["std_resid"] = std_resid
df["cooks_d"] = cooks
df["outlier_resid"] = np.abs(std_resid) > 3                 # HE: הסף שהוחלט ב-o_def
cooks_threshold = 4 / len(df)                                # HE: כלל אצבע מקובל, בחירה שלנו (לא החלטה בלוח)
df["influential"] = df["cooks_d"] > cooks_threshold
print("diagnostics pass three checks")
print("flagged by residual (|standardized residual| > 3):", int(df["outlier_resid"].sum()))
print("influential by Cook's distance (> 4/n =", round(cooks_threshold, 4), "):", int(df["influential"].sum()))

In [ ]:
# [P3] D2: the flagged laptops
# HE: המחיר הצפוי הוא exp של (לוג המחיר האמיתי פחות השארית). כך רואים כמה המחשב יקר או זול ממה שהמפרט מנבא.
df["expected_price"] = np.exp(df["log_price"] - resid)
cols = ["Company", "Product", "TypeName", "RAM (GB)", "Price (Euro)", "expected_price", "std_resid", "cooks_d"]
flagged = df.loc[df["outlier_resid"], cols].sort_values("std_resid")
print(flagged.round(2).to_string())
print()
print("flagged by both Tukey (log price) and residual:", int((df["outlier_tukey_log"] & df["outlier_resid"]).sum()))
print("flagged high (more expensive than expected):", int((df["std_resid"] > 3).sum()),
      "| flagged low:", int((df["std_resid"] < -3).sum()))
print("flagged by company:", flagged["Company"].value_counts().to_dict())
print()
print("most influential laptops (largest Cook's distance):")
print(df.nlargest(5, "cooks_d")[cols].round(2).to_string())

## 4. Unusual observations that are not about price

Decision `o_other`: three further checks. They tell us whether the unusual laptops are real machines or suspicious data.

### 4a. Physical fit: weight against screen size and type

A laptop's weight depends mainly on its screen size and on its type (an ultrabook is light, a gaming laptop is heavy). We fit a regression of the weight in kg on the screen size and the laptop type, and flag a laptop when the absolute value of its standardized residual is above 3, the same rule as for the price (the board did not fix this rule for weight, so it is our choice). A flagged laptop is not an error by itself: a heavy gaming laptop or a very light tablet-like laptop can be real.

In [ ]:
# [P3] D4a: physical fit, weight against screen size and type
# HE: אותה פונקציית אבחון כמו במחיר, אבל המשתנה המוסבר הוא המשקל, והמסבר הוא גודל המסך וסוג המחשב.
X_phys = design_matrix(df, ["Inches"], ["TypeName"])
_, std_weight, _, _ = residual_diagnostics(X_phys, df["Weight (kg)"])
beta_w, _ = ols_fit(X_phys, df["Weight (kg)"])
df["std_resid_weight"] = std_weight
df["expected_weight"] = X_phys.to_numpy() @ beta_w.to_numpy()
df["outlier_weight"] = np.abs(std_weight) > 3

wcols = ["Company", "Product", "TypeName", "Inches", "Weight (kg)", "expected_weight", "std_resid_weight"]
print("laptops flagged by weight:", int(df["outlier_weight"].sum()))
print(df.loc[df["outlier_weight"], wcols].sort_values("std_resid_weight").round(2).to_string())
print()
print("the two examples named on the board:")
print(df.loc[df["Product"].str.contains("G703|Yoga Book"), wcols].drop_duplicates("Product").round(2).to_string())

### 4b. Rare configurations and leverage

A laptop with an unusual configuration can have a large pull on a regression, even if its price is not unusual: this is called **leverage**. We call a configuration rare when the laptop has 24 GB of RAM or more, or when its CPU model or its GPU model appears in no other laptop in the table. We compare the leverage (from the specification model of section 3) of these laptops with the rest. As a rule of thumb, a leverage above `2 * k / n` is high, where `k` is the number of coefficients of the model and `n` the number of laptops; this threshold is our choice.

In [ ]:
# [P3] D4b: rare configurations and their leverage
# HE: מסמנים שלושה סוגי תצורה נדירה: RAM של 24 ומעלה, ודגם מעבד או כרטיס מסך שמופיע במחשב אחד בלבד.
df["rare_ram"] = df["RAM (GB)"] >= 24
df["unique_cpu"] = df["CPU_Type"].map(df["CPU_Type"].value_counts()) == 1
df["unique_gpu"] = df["GPU_Type"].map(df["GPU_Type"].value_counts()) == 1
df["rare_config"] = df[["rare_ram", "unique_cpu", "unique_gpu"]].any(axis=1)
df["leverage"] = lev                      # HE: ה-leverage מהמודל של סעיף 3 (מפרט בלבד)
lev_threshold = 2 * X_spec.shape[1] / len(df)

summary = pd.DataFrame({
    "laptops": [df[c].sum() for c in ["rare_ram", "unique_cpu", "unique_gpu", "rare_config"]] + [(~df["rare_config"]).sum()],
    "mean leverage": [df.loc[df[c], "leverage"].mean() for c in ["rare_ram", "unique_cpu", "unique_gpu", "rare_config"]] + [df.loc[~df["rare_config"], "leverage"].mean()],
    "share above threshold": [(df.loc[df[c], "leverage"] > lev_threshold).mean() for c in ["rare_ram", "unique_cpu", "unique_gpu", "rare_config"]] + [(df.loc[~df["rare_config"], "leverage"] > lev_threshold).mean()],
}, index=["RAM 24 GB or more", "CPU model appears once", "GPU model appears once", "any rare configuration", "all other laptops"])
print("leverage threshold 2k/n =", round(lev_threshold, 4))
print(summary.round(3).to_string())
print()
print("the five laptops with the highest leverage:")
print(df.nlargest(5, "leverage")[["Company", "Product", "TypeName", "RAM (GB)", "Price (Euro)", "leverage", "std_resid"]].round(3).to_string())

### 4c. Price spread within the same configuration

Part 2 found laptops that are identical in every column except the price (the near-duplicate rows). For these, the price gap cannot be explained by any variable of the table, so it measures the **unexplained noise** in the price: listings, stores, dates. We compare this noise with the spread that our specification model leaves unexplained. If they are similar, the model leaves little room for improvement with the variables we have.

In [ ]:
# [P3] D4c: price spread within the same configuration
# HE: מקבצים לפי כל העמודות המקוריות חוץ מהמחיר. קבוצה עם 2 מחשבים או יותר היא אותה תצורה במחירים שונים.
key_cols = [c for c in df.columns[:15] if c != "Price (Euro)"]
groups_cfg = df.groupby(key_cols)["Price (Euro)"].agg(n="size", low="min", high="max")
same_cfg = groups_cfg[groups_cfg["n"] >= 2].copy()
same_cfg["ratio"] = same_cfg["high"] / same_cfg["low"]
assert same_cfg["n"].sum() == df["near_duplicate"].sum()      # HE: אותם 48 מחשבים ש-Part 2 סימן

print("configurations listed more than once:", len(same_cfg), "(laptops:", int(same_cfg["n"].sum()), ")")
print("price ratio high/low: median", round(same_cfg["ratio"].median(), 2), "| largest", round(same_cfg["ratio"].max(), 2))
print(same_cfg.reset_index().nlargest(3, "ratio")[["Company", "Product", "low", "high", "ratio"]].round(2).to_string())

# HE: רעש "טהור": סטיית התקן של לוג המחיר בתוך אותה תצורה (שונות בתוך הקבוצות, מאוחדת).
dup = df.loc[df["near_duplicate"]]
deviation = dup["log_price"] - dup.groupby(key_cols)["log_price"].transform("mean")
pooled_noise_sd = np.sqrt((deviation ** 2).sum() / (len(dup) - len(same_cfg)))
# HE: מה שמודל המפרט משאיר בלי הסבר: סטיית התקן של השאריות שלו.
model_sd = np.sqrt((resid ** 2).sum() / (len(df) - X_spec.shape[1]))
print()
print(f"spread of log price inside the same configuration: {pooled_noise_sd:.3f}")
print(f"spread of log price left unexplained by the specification model: {model_sd:.3f}")

## 5. Do the unusual observations change the results?

Decision `o_treat`: we **keep** the unusual observations in the main analysis, and we show what would change if we removed them. For each scenario we compute five numbers: the mean and the median price, Pearson and Spearman between RAM and price, and the regression coefficient of RAM (the percent change in price for a doubling of RAM, from the full model of question 1). The mean and Pearson are sensitive to extreme values; the median and Spearman use ranks and are not. We use Pearson on the raw values as well as on the log-log scale, to show how much the scale matters.

The scenarios are: all laptops; without the laptop flagged by the Tukey fences (1 laptop); without the laptops flagged by the residuals (7); without both (8 laptops in total); without the single 64 GB laptop; and without all laptops with 24 GB or more of RAM (21 laptops), which are the high-leverage laptops of section 4b.

In [ ]:
# [P3] D5: key numbers with and without the unusual observations
# HE: אותם ארבעה-חמישה מדדים, מחושבים על תת-קבוצה של השורות. ההחלטה על מי להוציא היא רק לצורך ההשוואה.
num5, cat5 = models["5 + brand"]

def key_numbers(d):
    beta, _ = ols_fit(design_matrix(d, num5, cat5), d["log_price"])
    return pd.Series({
        "laptops": len(d),
        "mean price (euro)": d["Price (Euro)"].mean(),
        "median price (euro)": d["Price (Euro)"].median(),
        "Pearson raw (GB vs euro)": stats.pearsonr(d["RAM (GB)"], d["Price (Euro)"]).statistic,
        "Pearson log-log": stats.pearsonr(d["RAM_log2"], d["log_price"]).statistic,
        "Spearman": stats.spearmanr(d["RAM (GB)"], d["Price (Euro)"]).statistic,
        "RAM effect, % per doubling": (np.exp(beta["RAM_log2"]) - 1) * 100,
    })

scenarios = {
    "all laptops": pd.Series(True, index=df.index),
    "without Tukey outlier (1)": ~df["outlier_tukey_log"],
    "without residual outliers (7)": ~df["outlier_resid"],
    "without both (8)": ~(df["outlier_tukey_log"] | df["outlier_resid"]),
    "without the 64 GB laptop (1)": df["RAM (GB)"] < 64,
    "without RAM 24+ GB (21)": ~df["rare_ram"],
}
before_after = pd.DataFrame({name: key_numbers(df.loc[mask]) for name, mask in scenarios.items()})
# HE: בדיקות: מספרי השורות תואמים לספירות הקודמות, וה-"all laptops" זהה לתוצאות שחישבנו בשאלה 1.
assert list(before_after.loc["laptops"].astype(int)) == [1275, 1274, 1268, 1267, 1274, 1254]
assert abs(before_after.loc["Spearman", "all laptops"] - spearman) < 1e-12
assert abs(before_after.loc["RAM effect, % per doubling", "all laptops"] - ram_price_adjusted.loc["5 + brand", "pct_change"]) < 1e-2
print(before_after.round(3).to_string())

## 6. Interpretation

**What we observed.** The Tukey fences on the log price (158 to 5,765 euro) flag one laptop, a Razer Blade Pro gaming laptop at 6,099 euro. The residual rule flags seven laptops whose price differs a lot from what their specification predicts, six more expensive and one cheaper, and none of them is the Razer. They are HP, Dell and Lenovo laptops, among them an HP ZBook 17, two ThinkPad P laptops and an Alienware 17 labelled `Notebook`. By weight, 31 laptops are flagged, almost all 17-inch gaming laptops of 4.2 to 4.7 kg, many of them repeated listings of one model. The 21 laptops with 24 GB of RAM or more have a higher leverage than the others. Laptops with the same configuration but different prices differ by a median ratio of 1.09 (the largest, a Lenovo Yoga Book, is 1.72).

**What the analysis supports.** Most unusual observations look like real machines and not like errors: heavy gaming laptops, expensive workstations and business laptops. The only candidates for an error are two Alienware 17 rows: one has a 15.6 inch screen, while the other rows of the same model have 17.3 inches and the same weight, and one is labelled `Notebook`, while the other rows are `Gaming`. The second one is also flagged by the residual rule, because the model expects a low price for a `Notebook`. This is a hypothesis from comparing rows of the same model, and we did not check it with an external source. The main results do not depend on the unusual observations: removing the flagged laptops changes the median price by at most 9 euro, Spearman by 0.001, and the RAM effect from 35.1 to 32.6 percent per doubling. The mean and the raw Pearson correlation move more than the median and Spearman, as expected. The largest change is when the 21 laptops with 24 GB or more are removed (raw Pearson from 0.740 to 0.711, RAM effect 32.1 percent), because they are the end of the range. Since the picture does not change, we keep all laptops in the analysis (decision `o_treat`).

**What we cannot conclude.** We cannot say that a flagged laptop is an error: a price can be extreme and real, and a price far from the prediction may reflect a product line that our model does not capture (it has no brand and no graphics card). The residual rule depends on the model we chose and on the threshold 3, which is a convention; the weight model is simple, and the number of weight outliers shows that its residuals are not normal. The spread of 0.124 in the log price inside one configuration rests on only 24 pairs of laptops. Finally, we do not know when or where the prices were collected.